# 01 - Data Ingestion

Load sample PPG data, apply a clear schema, save the raw Delta table, and inspect basic data quality.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

spark.conf.set('spark.sql.shuffle.partitions', '8')

try:
    dbutils.widgets.text('input_csv_path', 'file:/Workspace/Users/YOUR_EMAIL/CardioTwin_AI_Databricks/data/sample_ppg_data.csv')
    input_csv_path = dbutils.widgets.get('input_csv_path')
except Exception:
    input_csv_path = '../data/sample_ppg_data.csv'

schema = StructType([
    StructField('patient_id', StringType(), False),
    StructField('window_id', StringType(), False),
    StructField('timestamp', TimestampType(), True),
    StructField('sampling_rate', IntegerType(), True),
    StructField('ppg_values', StringType(), True),
    StructField('cardiovascular_status', IntegerType(), True),
    StructField('systolic_bp', DoubleType(), True),
    StructField('diastolic_bp', DoubleType(), True),
])

raw_df = (
    spark.read
    .option('header', True)
    .schema(schema)
    .csv(input_csv_path)
)

display(raw_df.limit(10))

raw_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cardio_ppg_raw')

print('Saved Delta table: cardio_ppg_raw')
print(f'Rows loaded: {raw_df.count()}')

numeric_columns = ['sampling_rate', 'cardiovascular_status', 'systolic_bp', 'diastolic_bp']
display(raw_df.select(numeric_columns).summary())

missing_counts = raw_df.select([
    F.sum(F.col(column_name).isNull().cast('int')).alias(column_name)
    for column_name in raw_df.columns
])
display(missing_counts)

display(spark.sql('DESCRIBE TABLE cardio_ppg_raw'))
